# Clase 8 — Lectura y limpieza de documentos

En esta clase vamos a transformar documentos reales en texto limpio para usarlo después en el chatbot con RAG.

**Importante:** esta clase no entrena un modelo y todavía no responde con RAG completo. El objetivo es preparar los documentos para que en las próximas clases puedan fragmentarse, buscarse y usarse como contexto.

## Qué hacemos hoy

Al finalizar la clase deberías poder:

1. Entender cómo deberían ser las fuentes de conocimiento de tu chatbot.
2. Cargar archivos PDF, DOCX, TXT, MD o ZIP.
3. Extraer texto de esos documentos.
4. Detectar si un archivo no sirve o necesita revisión.
5. Limpiar el texto extraído.
6. Guardar una base documental en `documentos_proyecto.json` para usarla en la clase siguiente.
7. Generar un `resumen_documentos_clase8.csv` solo como control de lectura, no como fuente principal del chatbot.

## Ubicación de esta clase dentro del proyecto

| Clase | Trabajo principal | Resultado |
|---|---|---|
| Clase 7 | Definir y evaluar fuentes | Saber qué documentos necesita el chatbot |
| Clase 8 | Leer y limpiar documentos | Crear una base documental limpia |
| Clase 9 | Fragmentar documentos | Dividir textos en partes buscables |
| Clases siguientes | Embeddings + búsqueda + RAG | Responder usando fuentes |

Si todavía no hiciste tus fuentes, no pasa nada: hoy primero vamos a repasar cómo deberían ser y después practicamos la lectura con archivos demo o propios.

# 0. Guía rápida: cómo deberían ser las fuentes

Una fuente es un documento desde donde el chatbot va a tomar información.

Puede ser:

- un reglamento;
- una guía de inscripción;
- una página de preguntas frecuentes;
- un protocolo;
- un manual;
- un documento institucional;
- un archivo TXT o Markdown con información ordenada.

Una buena fuente debería tener:

- título claro;
- secciones o subtítulos;
- información concreta;
- fechas o condiciones explícitas si corresponde;
- datos confiables;
- texto seleccionable, no solo imagen;
- contenido relacionado con el objetivo del chatbot.

Una fuente problemática sería:

- una captura de pantalla sin texto extraíble;
- información copiada sin origen claro;
- rumores o datos no oficiales;
- texto mezclado, desordenado o incompleto;
- documentos que no tienen relación con el tema del chatbot.

## ¿Por qué no trabajamos directamente con el PDF?

El PDF, DOCX o TXT es la **fuente original**.

Pero para construir un chatbot con RAG, necesitamos convertir esos archivos en texto procesable.

Por eso en esta clase generamos dos salidas:

| Archivo generado | Para qué sirve |
|---|---|
| `documentos_proyecto.json` | Es la base documental limpia. La vamos a usar en la clase 9 para fragmentar documentos. |
| `resumen_documentos_clase8.csv` | Es solo un resumen para revisar qué se leyó, cuántos caracteres salieron y si algún archivo necesita revisión. |

El CSV **no reemplaza al PDF** y tampoco es la fuente principal del chatbot. Es una planilla de control para humanos.

In [ ]:
# No necesitamos GPU en esta clase.
!pip -q install pandas pypdf python-docx

In [ ]:
from pathlib import Path
import json
import re
import zipfile
from datetime import datetime

import pandas as pd
from pypdf import PdfReader
from docx import Document

pd.set_option("display.max_colwidth", 300)
pd.set_option("display.max_columns", None)

print("Entorno listo")

# 1. Datos del proyecto

Completá estos datos con tu temática real.

In [ ]:
proyecto = {
    "nombre_chatbot": "MiChatbot",
    "tematica": "Orientación académica",
    "usuario_objetivo": "Estudiantes e ingresantes",
    "descripcion": "Chatbot que responde consultas usando documentos confiables."
}

pd.DataFrame([proyecto])

# 2. Plan mínimo de fuentes

Si no trajiste tus documentos, usá esta tabla para pensar qué debería tener tu chatbot.

No es obligatorio que estos datos estén dentro del PDF. Esta tabla sirve para organizar el trabajo.

In [ ]:
plan_fuentes = [
    {
        "fuente_necesaria": "Reglamento o normativa",
        "que_deberia_responder": "Condiciones, permisos, límites, requisitos o reglas",
        "estado": "pendiente",
        "archivo_real": ""
    },
    {
        "fuente_necesaria": "Guía o instructivo",
        "que_deberia_responder": "Pasos para realizar un trámite o proceso",
        "estado": "pendiente",
        "archivo_real": ""
    },
    {
        "fuente_necesaria": "Preguntas frecuentes",
        "que_deberia_responder": "Consultas comunes de los usuarios",
        "estado": "pendiente",
        "archivo_real": ""
    }
]

df_plan_fuentes = pd.DataFrame(plan_fuentes)
df_plan_fuentes

# 3. Subir archivos

Subí tus documentos reales o los PDFs demo.

Formatos aceptados:

- `.pdf`
- `.docx`
- `.txt`
- `.md`
- `.zip` con varios archivos adentro

Si un PDF está escaneado como imagen, probablemente no se pueda extraer texto sin OCR.

In [ ]:
from google.colab import files

archivos_subidos = files.upload()

rutas_subidas = [Path(nombre) for nombre in archivos_subidos.keys()]
rutas_subidas

# 4. Preparar rutas de archivos

Si subiste un ZIP, esta celda lo descomprime y busca documentos adentro.

In [ ]:
FORMATOS_SOPORTADOS = [".pdf", ".docx", ".txt", ".md"]

carpeta_extraidos = Path("fuentes_extraidas")
carpeta_extraidos.mkdir(exist_ok=True)

rutas_archivos = []

for ruta in rutas_subidas:
    extension = ruta.suffix.lower()

    if extension == ".zip":
        with zipfile.ZipFile(ruta, "r") as zip_ref:
            zip_ref.extractall(carpeta_extraidos)

        for archivo in carpeta_extraidos.rglob("*"):
            if archivo.is_file() and archivo.suffix.lower() in FORMATOS_SOPORTADOS:
                rutas_archivos.append(archivo)

    elif extension in FORMATOS_SOPORTADOS:
        rutas_archivos.append(ruta)

rutas_archivos = sorted(rutas_archivos, key=lambda p: p.name.lower())

print("Archivos detectados:")
for ruta in rutas_archivos:
    print("-", ruta)

# 5. Funciones para extraer texto

In [ ]:
def extraer_texto_pdf(ruta: str) -> str:
    lector = PdfReader(ruta)
    paginas = []

    for numero, pagina in enumerate(lector.pages, start=1):
        texto = pagina.extract_text() or ""
        if texto.strip():
            paginas.append(f"Página {numero}\n{texto}")

    return "\n\n".join(paginas)


def extraer_texto_docx(ruta: str) -> str:
    documento = Document(ruta)
    parrafos = [p.text for p in documento.paragraphs if p.text.strip()]
    return "\n".join(parrafos)


def extraer_texto_txt_md(ruta: str) -> str:
    with open(ruta, "r", encoding="utf-8", errors="ignore") as archivo:
        return archivo.read()


def extraer_texto_archivo(ruta: Path) -> str:
    extension = ruta.suffix.lower()

    if extension == ".pdf":
        return extraer_texto_pdf(str(ruta))

    if extension == ".docx":
        return extraer_texto_docx(str(ruta))

    if extension in [".txt", ".md"]:
        return extraer_texto_txt_md(str(ruta))

    raise ValueError(f"Formato no soportado: {extension}")

# 6. Limpieza básica

La limpieza no debe borrar información importante. Solo buscamos dejar el texto más legible para trabajar en la próxima clase.

In [ ]:
def limpiar_documento(texto: str) -> str:
    texto = texto.replace("\x00", " ")
    texto = texto.replace("\r", "\n")

    # Quita espacios repetidos dentro de cada línea.
    lineas = []
    for linea in texto.split("\n"):
        linea = re.sub(r"\s+", " ", linea).strip()
        if linea:
            lineas.append(linea)

    texto = "\n".join(lineas)

    # Reduce saltos de línea excesivos.
    texto = re.sub(r"\n{3,}", "\n\n", texto)

    return texto.strip()


def obtener_muestra_util(texto: str, max_caracteres: int = 350) -> str:
    """
    Muestra una parte más útil del documento.
    Evita quedarse solo con frases introductorias repetidas de los PDFs demo.
    """
    frases_a_omitir = [
        "Documento ficticio creado para la Clase 7",
        "No tiene validez institucional real",
        "Página 1"
    ]

    lineas_utiles = []
    for linea in texto.split("\n"):
        linea_limpia = linea.strip()
        if not linea_limpia:
            continue
        if any(frase.lower() in linea_limpia.lower() for frase in frases_a_omitir):
            continue
        lineas_utiles.append(linea_limpia)

    muestra = " ".join(lineas_utiles)
    if not muestra:
        muestra = texto.replace("\n", " ")

    return muestra[:max_caracteres]


def sugerir_uso(nombre_archivo: str, texto: str) -> tuple[str, str]:
    nombre = nombre_archivo.lower()
    texto_bajo = texto.lower()

    if len(texto.strip()) < 200:
        return "revisar", "Se extrajo poco texto. Puede ser un PDF escaneado o un archivo incompleto."

    if "no oficial" in nombre or "descartar" in nombre or "fuente no oficial" in texto_bajo:
        return "descartar o usar solo como ejemplo negativo", "La fuente parece no oficial o fue creada para mostrar un caso problemático."

    if "documento ficticio" in texto_bajo:
        return "usar solo para práctica", "Es un documento demo. Sirve para practicar, pero no para un chatbot institucional real."

    return "usar como fuente", "Tiene texto extraíble y puede servir para la base documental."

# 7. Procesar documentos

Esta celda extrae, limpia y revisa cada documento.

La columna `muestra_util` no busca responder todavía. Solo permite ver si el texto se leyó bien.

In [ ]:
documentos = []
resumen = []

for ruta in rutas_archivos:
    try:
        texto_crudo = extraer_texto_archivo(ruta)
        texto_limpio = limpiar_documento(texto_crudo)
        uso_sugerido, motivo = sugerir_uso(ruta.name, texto_limpio)

        documentos.append({
            "fuente": ruta.name,
            "tipo": ruta.suffix.lower().replace(".", ""),
            "texto": texto_limpio
        })

        resumen.append({
            "fuente": ruta.name,
            "tipo": ruta.suffix.lower(),
            "caracteres_extraidos": len(texto_limpio),
            "estado_lectura": "texto extraíble" if len(texto_limpio) > 200 else "revisar: poco texto",
            "uso_sugerido": uso_sugerido,
            "motivo": motivo,
            "muestra_util": obtener_muestra_util(texto_limpio)
        })

    except Exception as error:
        resumen.append({
            "fuente": ruta.name,
            "tipo": ruta.suffix.lower(),
            "caracteres_extraidos": 0,
            "estado_lectura": f"error: {error}",
            "uso_sugerido": "revisar",
            "motivo": "No se pudo leer el archivo.",
            "muestra_util": ""
        })


df_resumen = pd.DataFrame(resumen)
df_resumen

## Cómo leer la tabla anterior

- `caracteres_extraidos`: indica cuánto texto se pudo obtener.
- `estado_lectura`: muestra si el archivo pudo leerse.
- `uso_sugerido`: orienta si conviene usar, revisar o descartar la fuente.
- `muestra_util`: muestra una parte del contenido, evitando quedarse solo con frases repetidas de presentación.

Si varios documentos empiezan con la misma frase, no significa que sean iguales: puede ser solo una portada o encabezado repetido.

# 8. Revisar una muestra completa

Cambiá el número de `indice` para revisar cada documento.

In [ ]:
indice = 0

if documentos:
    print("Fuente:", documentos[indice]["fuente"])
    print("Caracteres:", len(documentos[indice]["texto"]))
    print("-" * 80)
    print(documentos[indice]["texto"][:2000])
else:
    print("No hay documentos procesados.")

# 9. Decisión sobre las fuentes

Completá esta tabla después de revisar los documentos.

La idea es que no todo lo que se sube al Colab debe entrar al chatbot. Algunas fuentes pueden necesitar revisión o descartarse.

In [ ]:
decisiones = []

for item in resumen:
    decisiones.append({
        "fuente": item["fuente"],
        "decision": item["uso_sugerido"],
        "justificacion": item["motivo"]
    })

df_decisiones = pd.DataFrame(decisiones)
df_decisiones

# 10. Guardar la base documental

Ahora guardamos los resultados.

Archivos que se generan:

1. `documentos_proyecto.json`  
   Base documental limpia. **Este es el archivo importante para la clase 9.**

2. `resumen_documentos_clase8.csv`  
   Resumen para revisar qué archivos se pudieron leer. **No es la base principal del chatbot.**

3. `plan_fuentes_clase8.csv`  
   Guía para organizar qué fuentes necesita el proyecto.

4. `decisiones_fuentes_clase8.csv`  
   Registro de qué fuentes se usan, revisan o descartan.

In [ ]:
salida_json = "documentos_proyecto.json"
salida_resumen_csv = "resumen_documentos_clase8.csv"
salida_plan_csv = "plan_fuentes_clase8.csv"
salida_decisiones_csv = "decisiones_fuentes_clase8.csv"

with open(salida_json, "w", encoding="utf-8") as archivo:
    json.dump(documentos, archivo, ensure_ascii=False, indent=2)

df_resumen.to_csv(salida_resumen_csv, index=False, encoding="utf-8-sig")
df_plan_fuentes.to_csv(salida_plan_csv, index=False, encoding="utf-8-sig")
df_decisiones.to_csv(salida_decisiones_csv, index=False, encoding="utf-8-sig")

print("Archivos generados:")
print("-", salida_json)
print("-", salida_resumen_csv)
print("-", salida_plan_csv)
print("-", salida_decisiones_csv)

# 11. Crear ZIP de entrega

El ZIP sirve para entregar todo junto.

La entrega principal es `documentos_proyecto.json`, porque en la próxima clase vamos a usarlo para fragmentar los documentos.

In [ ]:
fecha = datetime.now().strftime("%Y%m%d_%H%M")
nombre_zip = f"entrega_clase8_lectura_limpieza_documentos_{fecha}.zip"

archivos_para_zip = [
    salida_json,
    salida_resumen_csv,
    salida_plan_csv,
    salida_decisiones_csv
]

with zipfile.ZipFile(nombre_zip, "w") as zipf:
    for archivo in archivos_para_zip:
        if Path(archivo).exists():
            zipf.write(archivo)

print("ZIP generado:", nombre_zip)

In [ ]:
files.download(nombre_zip)

# Entrega de la clase

Entregar el ZIP generado al final.

Debe incluir:

1. `documentos_proyecto.json`.
2. `resumen_documentos_clase8.csv`.
3. `plan_fuentes_clase8.csv`.
4. `decisiones_fuentes_clase8.csv`.

Además, cada grupo debe poder explicar:

- qué documentos usó;
- cuáles sirven como fuente;
- cuáles habría que revisar;
- si algún PDF no pudo leerse bien;
- por qué el JSON será útil para la clase 9.